In [2]:
# ==========================================
# 1. 라이브러리 및 모듈 불러오기
# ==========================================
import numpy as np
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier

# 데이터 분리 및 교차 검증 관련 모듈
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# 성능 평가 관련 평가지표 모듈
from sklearn.metrics import (
    confusion_matrix, 
    accuracy_score, 
    classification_report, 
    roc_auc_score, 
    mean_squared_error
)

In [3]:
# ==========================================
# 2. 데이터셋 로드 및 준비
# ==========================================
# Scikit-Learn에서 제공하는 유방암(Breast Cancer) 데이터셋 사용
data = datasets.load_breast_cancer()
X = data.data    # 특성 데이터 (Feature)
y = data.target  # 타겟 레이블 (Target, 0: 악성, 1: 양성)

print(f"전체 데이터 구조 - 특성(X): {X.shape}, 타겟(y): {y.shape}\n")

전체 데이터 구조 - 특성(X): (569, 30), 타겟(y): (569,)



In [4]:
# ==========================================
# 3. [방법 A] 홀드아웃 (Hold-out) 교차 검증
# ==========================================
print("-" * 50)
print(" 방법 A: 홀드아웃 (Hold-out) 교차 검증 진행")
print("-" * 50)

# 데이터를 8:2 비율로 훈련셋과 테스트셋으로 1회 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 의사결정나무(Decision Tree) 모델 생성 및 학습
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# 테스트 데이터셋을 통한 예측
y_pred = clf.predict(X_test)

# 홀드아웃 모델 성능 평가 출력
print("[Confusion Matrix]")
print(confusion_matrix(y_test, y_pred))
print(f"\nAccuracy: {accuracy_score(y_test, y_pred, normalize=True):.4f}")
print(f"AUC Score: {roc_auc_score(y_test, y_pred):.4f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred):.4f}")
print("\n[Classification Report]")
print(classification_report(y_test, y_pred))

--------------------------------------------------
 방법 A: 홀드아웃 (Hold-out) 교차 검증 진행
--------------------------------------------------
[Confusion Matrix]
[[40  3]
 [ 3 68]]

Accuracy: 0.9474
AUC Score: 0.9440
Mean Squared Error: 0.0526

[Classification Report]
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        43
           1       0.96      0.96      0.96        71

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



In [5]:
# ==========================================
# 4. [방법 B] Stratified K-Fold 교차 검증 (순차적 분할)
# ==========================================
print("\n" + "-" * 50)
print(" 방법 B: Stratified K-Fold 교차 검증 (순차적 분할)")
print("-" * 50)

# 10개의 폴드로 분할하는 객체 생성 (데이터를 섞지 않고 순서대로 분할)
skf = StratifiedKFold(n_splits=10)

# 각 폴드별 데이터 인덱스 확인 (확인용 출력, 데이터가 많으므로 처음 1개 폴드만 예시 출력 가능)
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"[Fold {i+1}] Train set 인덱스 일부: {train_index[:5]}... | Test set 인덱스 일부: {test_index[:5]}...")
    if i == 2: # 출력이 너무 길어지는 것을 방지하기 위해 3개만 출력하고 끊습니다.
        print("... (이하 중략) ...")
        break

# 모델 평가 및 10개의 교차 검증 점수 출력
clf_kfold = DecisionTreeClassifier(random_state=42)
scores = cross_val_score(clf_kfold, X, y, cv=skf)

print("\n[K-Fold 교차 검증 결과]")
print(f"각 폴드별 정확도(Scores): {scores}")
print(f"평균 정확도(Average Accuracy): {scores.mean():.4f}")


--------------------------------------------------
 방법 B: Stratified K-Fold 교차 검증 (순차적 분할)
--------------------------------------------------
[Fold 1] Train set 인덱스 일부: [25 26 27 28 29]... | Test set 인덱스 일부: [0 1 2 3 4]...
[Fold 2] Train set 인덱스 일부: [0 1 2 3 4]... | Test set 인덱스 일부: [25 26 27 28 29]...
[Fold 3] Train set 인덱스 일부: [0 1 2 3 4]... | Test set 인덱스 일부: [54 56 57 62 64]...
... (이하 중략) ...

[K-Fold 교차 검증 결과]
각 폴드별 정확도(Scores): [0.92982456 0.89473684 0.92982456 0.89473684 0.98245614 0.89473684
 0.89473684 0.94736842 0.92982456 0.98214286]
평균 정확도(Average Accuracy): 0.9280


In [6]:
# ==========================================
# 5. [방법 C] Stratified K-Fold 교차 검증 (Shuffle 추가)
# ==========================================
print("\n" + "-" * 50)
print(" 방법 C: Stratified K-Fold 교차 검증 (Shuffle=True 추가)")
print("-" * 50)

# 데이터를 무작위로 섞은(Shuffle) 뒤 10개 폴드로 분할하는 객체 생성
skf_sh = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 각 폴드별 데이터 인덱스 확인 (무작위로 섞인 인덱스 확인 가능)
for i, (train_index, test_index) in enumerate(skf_sh.split(X, y)):
    print(f"[Fold {i+1} - Shuffled] Train 인덱스 일부: {train_index[:5]}... | Test 인덱스 일부: {test_index[:5]}...")
    if i == 2:
        print("... (이하 중략) ...")
        break

# Shuffle 기반 모델 평가 및 점수 출력
clf_kfold_sh = DecisionTreeClassifier(random_state=42)
scores_sh = cross_val_score(clf_kfold_sh, X, y, cv=skf_sh)

print("\n[Shuffled K-Fold 교차 검증 결과]")
print(f"각 폴드별 정확도(Scores): {scores_sh}")
print(f"평균 정확도(Average Accuracy): {scores_sh.mean():.4f}")


--------------------------------------------------
 방법 C: Stratified K-Fold 교차 검증 (Shuffle=True 추가)
--------------------------------------------------
[Fold 1 - Shuffled] Train 인덱스 일부: [0 1 2 3 4]... | Test 인덱스 일부: [ 8 14 17 31 55]...
[Fold 2 - Shuffled] Train 인덱스 일부: [1 2 3 4 5]... | Test 인덱스 일부: [ 0 29 65 70 75]...
[Fold 3 - Shuffled] Train 인덱스 일부: [0 1 2 3 5]... | Test 인덱스 일부: [ 4  6 19 21 24]...
... (이하 중략) ...

[Shuffled K-Fold 교차 검증 결과]
각 폴드별 정확도(Scores): [0.9122807  0.92982456 0.96491228 0.89473684 0.89473684 0.96491228
 0.92982456 0.9122807  0.92982456 0.92857143]
평균 정확도(Average Accuracy): 0.9262
